<img src="images/logo.jpg" style="display: block; margin-left: auto; margin-right: auto;" alt="לוגו הקורס">

# <p style="text-align: right; direction: rtl; float: right;">יום 5 - תרגולים SQL</p>

<p style="text-align: right; direction: rtl; float: right; clear: both;">
בתרגולים של היום נעבוד עם מסד נתונים של <strong>חנות מכולת קטנה</strong>: טבלת מוצרים וטבלת הזמנות.<br>
התרגילים מסודרים מהקל אל הקשה:<br>
🔹 תרגילים 1-2: שליפה, סינון ומיון<br>
🔹 תרגילים 3-4: UPDATE, DELETE, GROUP BY ו-HAVING<br>
🔹 תרגילים 5-6: JOIN, ויבוא קובץ CSV ושמירה ל-JSON<br>
🔹 אתגר: מערכת ספרייה עם מחלקה<br><br>
בכל תרגיל יש קוד פתיחה שאפשר להמשיך ממנו, ורמזים.
</p>

<div class="align-center" style="display: flex; text-align: right; direction: rtl; clear: both;">
    <div style="display: flex; width: 10%; float: right; clear: both;">
        <img src="images/tip.png" style="height: 50px !important;" alt="טיפ">
    </div>
    <div style="width: 90%">
        <p style="text-align: right; direction: rtl; float: right; clear: both;">
        <strong>חשוב להבין</strong>:<br>
            אם קיבלתם שגיאה (למשל <code>no such table</code>, או <code>Cannot operate on a closed database</code>) -
הריצו מחדש את התא <strong>"הכנת מסד הנתונים"</strong> שלמטה, ואז את התרגיל שאתם עובדים עליו.
התא הזה בונה את הטבלאות מחדש, ולכן גם מאפס שינויים של תרגילי UPDATE ו-DELETE.
        </p>
    </div>
</div>

## <p style="text-align: right; direction: rtl; float: right; clear: both;">הכנת מסד הנתונים</p>

<p style="text-align: right; direction: rtl; float: right; clear: both;">
<strong>הריצו את התא הבא לפני שמתחילים.</strong> הוא יוצר מסד נתונים <code>shop.db</code> עם שתי טבלאות:<br>
🔹 <code>products</code> - מוצרים: <code>id, name, category, price, stock</code> (מלאי)<br>
🔹 <code>orders</code> - הזמנות: <code>id, customer, product_id, quantity, order_date</code><br>
העמודה <code>product_id</code> בהזמנות מצביעה על <code>id</code> בטבלת המוצרים.
</p>

In [ ]:
import sqlite3
import csv
import json
import os

try:
    conn.close()
except NameError:
    pass

conn = sqlite3.connect('shop.db')
cursor = conn.cursor()

cursor.execute('DROP TABLE IF EXISTS orders')
cursor.execute('DROP TABLE IF EXISTS products')

cursor.execute('''
CREATE TABLE products (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    category TEXT,
    price REAL,
    stock INTEGER
)
''')

cursor.execute('''
CREATE TABLE orders (
    id INTEGER PRIMARY KEY,
    customer TEXT NOT NULL,
    product_id INTEGER,
    quantity INTEGER,
    order_date TEXT
)
''')

products = [
    (1, 'חלב 3%', 'מוצרי חלב', 6.5, 40),
    (2, 'גבינה צהובה', 'מוצרי חלב', 28.0, 15),
    (3, 'לחם אחיד', 'מאפים', 8.0, 30),
    (4, 'חלה', 'מאפים', 15.0, 12),
    (5, 'עגבניות', 'ירקות', 7.0, 50),
    (6, 'מלפפונים', 'ירקות', 5.5, 0),
    (7, 'אורז', 'מכולת', 12.0, 25),
    (8, 'פסטה', 'מכולת', 9.0, 0),
    (9, 'שמן זית', 'מכולת', 38.0, 10),
    (10, 'קפה', 'משקאות', 24.0, 18),
    (11, 'מיץ תפוזים', 'משקאות', 14.0, 22),
    (12, 'שוקולד', 'חטיפים', 11.0, 35),
]

orders = [
    (1, 'דנה', 1, 2, '2023-12-20'),
    (2, 'דנה', 3, 1, '2024-01-05'),
    (3, 'אורי', 1, 3, '2024-01-06'),
    (4, 'אורי', 7, 2, '2024-01-08'),
    (5, 'מיכל', 10, 1, '2024-01-10'),
    (6, 'מיכל', 2, 1, '2024-01-12'),
    (7, 'דנה', 9, 1, '2024-01-15'),
    (8, 'יוסי', 5, 4, '2024-01-16'),
    (9, 'יוסי', 1, 2, '2024-01-16'),
    (10, 'אורי', 4, 2, '2024-01-20'),
    (11, 'מיכל', 11, 3, '2024-01-22'),
    (12, 'דנה', 1, 1, '2024-01-25'),
    (13, 'יוסי', 10, 2, '2023-11-30'),
    (14, 'אורי', 5, 1, '2024-02-01'),
]

cursor.executemany('INSERT INTO products VALUES (?, ?, ?, ?, ?)', products)
cursor.executemany('INSERT INTO orders VALUES (?, ?, ?, ?, ?)', orders)
conn.commit()

print(f"נוצרו {len(products)} מוצרים ו-{len(orders)} הזמנות ✅")

<p style="text-align: right; direction: rtl; float: right; clear: both;">
נסתכל על כמה שורות מכל טבלה כדי להכיר את הנתונים:
</p>

In [ ]:
cursor.execute('SELECT * FROM products LIMIT 3')
print("products:", cursor.fetchall())

cursor.execute('SELECT * FROM orders LIMIT 3')
print("orders:  ", cursor.fetchall())

## <p style="text-align: right; direction: rtl; float: right; clear: both;">תרגיל 1: שליפה, סינון ומיון (קל)</p>

<div class="align-center" style="display: flex; text-align: right; direction: rtl; clear: both;">
    <div style="display: flex; width: 10%; float: right; clear: both;">
        <img src="images/exercise.svg" style="height: 50px !important;" alt="תרגול">
    </div>
    <div style="width: 90%">
        <p style="text-align: right; direction: rtl; float: right; clear: both;">
        <strong>תרגול</strong>:<br>
            הציגו את כל המוצרים בקטגוריה "מכולת": שם ומחיר, ממוינים מהיקר לזול.<br>
הדפיסו כל מוצר בשורה נפרדת בצורה: <code>שמן זית - ₪38.0</code>
        </p>
    </div>
</div>

<p style="text-align: right; direction: rtl; float: right; clear: both;">
💡 <strong>רמז:</strong> <code>SELECT name, price FROM products WHERE category = ? ORDER BY price DESC</code> - ואל תשכחו להעביר את הקטגוריה כ-tuple: <code>(category,)</code>.
</p>

In [ ]:
# פתרו כאן:
category = 'מכולת'  # שנו כאן

# cursor.execute('SELECT ... FROM products WHERE category = ? ORDER BY ...', (category,))
# for name, price in cursor.fetchall():
#     print(...)

## <p style="text-align: right; direction: rtl; float: right; clear: both;">תרגיל 2: פרמטרים וסינון (קל)</p>

<div class="align-center" style="display: flex; text-align: right; direction: rtl; clear: both;">
    <div style="display: flex; width: 10%; float: right; clear: both;">
        <img src="images/exercise.svg" style="height: 50px !important;" alt="תרגול">
    </div>
    <div style="width: 90%">
        <p style="text-align: right; direction: rtl; float: right; clear: both;">
        <strong>תרגול</strong>:<br>
            כתבו שתי שאילתות עם פרמטרים (<code>?</code>):
        </p>
    </div>
</div>

<ol style="text-align: right; direction: rtl; float: right; clear: both;">
    <li>כל המוצרים שמחירם בין <code>min_price</code> ל-<code>max_price</code> (כולל)</li>
    <li>כל המוצרים שאזלו מהמלאי (<code>stock = 0</code>) - הדפיסו רק את שמותיהם</li>
</ol>

<p style="text-align: right; direction: rtl; float: right; clear: both;">
💡 <strong>רמז:</strong> <code>BETWEEN ? AND ?</code> ו-tuple עם שני ערכים: <code>(min_price, max_price)</code>.
</p>

In [ ]:
# פתרו כאן:
min_price = 8    # שנו כאן
max_price = 15   # שנו כאן

# שאילתה 1: מוצרים בטווח מחירים


# שאילתה 2: מוצרים שאזלו

## <p style="text-align: right; direction: rtl; float: right; clear: both;">תרגיל 3: UPDATE ו-DELETE (בינוני)</p>

<div class="align-center" style="display: flex; text-align: right; direction: rtl; clear: both;">
    <div style="display: flex; width: 10%; float: right; clear: both;">
        <img src="images/exercise.svg" style="height: 50px !important;" alt="תרגול">
    </div>
    <div style="width: 90%">
        <p style="text-align: right; direction: rtl; float: right; clear: both;">
        <strong>תרגול</strong>:<br>
            בצעו שני שינויים בנתונים, ובכל אחד הציגו <strong>לפני</strong> ו<strong>אחרי</strong>:
        </p>
    </div>
</div>

<ol style="text-align: right; direction: rtl; float: right; clear: both;">
    <li>העלו ב-5% את המחיר של כל מוצרי החלב (קטגוריה "מוצרי חלב")</li>
    <li>מחקו את כל ההזמנות שנעשו לפני 1.1.2024 (<code>order_date &lt; '2024-01-01'</code>), והדפיסו כמה הזמנות נמחקו</li>
</ol>

<p style="text-align: right; direction: rtl; float: right; clear: both;">
💡 <strong>רמזים:</strong><br>
🔹 <code>UPDATE products SET price = ROUND(price * 1.05, 2) WHERE category = ?</code><br>
🔹 תאריכים בפורמט <code>YYYY-MM-DD</code> אפשר להשוות כמו טקסט: <code>order_date &lt; '2024-01-01'</code><br>
🔹 <strong>תמיד WHERE</strong>, ותמיד <code>conn.commit()</code><br>
🔹 אם הרצתם את התא פעמיים והמחירים התנפחו - הריצו שוב את תא "הכנת מסד הנתונים"
</p>

In [ ]:
# פתרו כאן:

# --- חלק א: העלאת מחירים ---
print("מחירי מוצרי החלב לפני:")
cursor.execute("SELECT name, price FROM products WHERE category = 'מוצרי חלב'")
print(cursor.fetchall())

# UPDATE ...

# print("אחרי:")

# --- חלק ב: מחיקת הזמנות ישנות ---
# cursor.execute("SELECT COUNT(*) FROM orders")  # כמה הזמנות יש לפני?
# DELETE ...
# print(f"נמחקו {cursor.rowcount} הזמנות")

## <p style="text-align: right; direction: rtl; float: right; clear: both;">תרגיל 4: GROUP BY ו-HAVING (בינוני)</p>

<div class="align-center" style="display: flex; text-align: right; direction: rtl; clear: both;">
    <div style="display: flex; width: 10%; float: right; clear: both;">
        <img src="images/exercise.svg" style="height: 50px !important;" alt="תרגול">
    </div>
    <div style="width: 90%">
        <p style="text-align: right; direction: rtl; float: right; clear: both;">
        <strong>תרגול</strong>:<br>
            חשבו כמה יחידות (<code>quantity</code>) הזמין כל לקוח בסך הכל.<br>
הציגו רק לקוחות שהזמינו <strong>5 יחידות או יותר</strong>, ממוינים מהגבוה לנמוך, בצורה: <code>אורי: 8 יחידות</code>
        </p>
    </div>
</div>

<p style="text-align: right; direction: rtl; float: right; clear: both;">
💡 <strong>רמז:</strong> <code>SELECT customer, SUM(quantity) AS total FROM orders GROUP BY customer HAVING SUM(quantity) &gt;= 5 ORDER BY total DESC</code>.<br>
אם מחקתם הזמנות ישנות בתרגיל 3, התוצאות ישתנו קצת - זה בסדר.
</p>

In [ ]:
# פתרו כאן:
min_units = 5  # שנו כאן

# cursor.execute('''
#     SELECT customer, SUM(quantity) AS total
#     FROM orders
#     GROUP BY ...
#     HAVING ...
#     ORDER BY ...
# ''', (min_units,))

## <p style="text-align: right; direction: rtl; float: right; clear: both;">תרגיל 5: JOIN (קשה)</p>

<div class="align-center" style="display: flex; text-align: right; direction: rtl; clear: both;">
    <div style="display: flex; width: 10%; float: right; clear: both;">
        <img src="images/exercise.svg" style="height: 50px !important;" alt="תרגול">
    </div>
    <div style="width: 90%">
        <p style="text-align: right; direction: rtl; float: right; clear: both;">
        <strong>תרגול</strong>:<br>
            חברו את הטבלאות <code>orders</code> ו-<code>products</code>:
        </p>
    </div>
</div>

<ol style="text-align: right; direction: rtl; float: right; clear: both;">
    <li>הציגו לכל הזמנה: שם הלקוח, שם המוצר, הכמות והסכום לתשלום (<code>price * quantity</code>)</li>
    <li>הציגו לכל מוצר את ההכנסה הכוללת ממנו (סכום ה-<code>price * quantity</code> של כל ההזמנות), ממוינת מהגבוהה לנמוכה</li>
    <li>הציגו את המוצרים ש<strong>אף פעם לא הוזמנו</strong> (רמז: LEFT JOIN, ובדיקה ש-<code>orders.id IS NULL</code>)</li>
</ol>

<p style="text-align: right; direction: rtl; float: right; clear: both;">
💡 <strong>רמזים:</strong><br>
🔹 <code>FROM orders AS o INNER JOIN products AS p ON o.product_id = p.id</code><br>
🔹 להכנסה לכל מוצר: <code>GROUP BY p.name</code> ו-<code>SUM(p.price * o.quantity)</code><br>
🔹 למוצרים שלא הוזמנו: <code>FROM products AS p LEFT JOIN orders AS o ON o.product_id = p.id WHERE o.id IS NULL</code>
</p>

In [ ]:
# פתרו כאן:

# חלק 1: הזמנות עם שם מוצר וסכום
# cursor.execute('''
#     SELECT o.customer, p.name, o.quantity, p.price * o.quantity AS total
#     FROM orders AS o
#     INNER JOIN products AS p ON ...
# ''')

# חלק 2: הכנסות לכל מוצר


# חלק 3: מוצרים שלא הוזמנו

## <p style="text-align: right; direction: rtl; float: right; clear: both;">תרגיל 6: מ-CSV למסד נתונים ול-JSON (קשה)</p>

<div class="align-center" style="display: flex; text-align: right; direction: rtl; clear: both;">
    <div style="display: flex; width: 10%; float: right; clear: both;">
        <img src="images/exercise.svg" style="height: 50px !important;" alt="תרגול">
    </div>
    <div style="width: 90%">
        <p style="text-align: right; direction: rtl; float: right; clear: both;">
        <strong>תרגול</strong>:<br>
            בתיקייה <code>resources</code> יש קובץ מכירות <code>sales.csv</code> עם העמודות: <code>date, product, category, quantity, price, total</code>.
        </p>
    </div>
</div>

<ol style="text-align: right; direction: rtl; float: right; clear: both;">
    <li>צרו טבלה <code>sales</code> במסד הנתונים (עם <code>DROP TABLE IF EXISTS</code> לפני, כדי שאפשר יהיה להריץ שוב)</li>
    <li>טענו אליה את כל השורות מהקובץ עם <code>csv.DictReader</code> ו-<code>executemany</code> (אל תשכחו להמיר מספרים עם <code>int()</code>)</li>
    <li>חשבו עם <code>GROUP BY</code> את סך המכירות (<code>SUM(total)</code>) לכל קטגוריה</li>
    <li>הפכו את התוצאה לרשימה של מילונים (<code>sqlite3.Row</code> או <code>dict(zip(...))</code>) ושמרו ל-<code>reports/sales_by_category.json</code></li>
</ol>

<p style="text-align: right; direction: rtl; float: right; clear: both;">
💡 <strong>רמזים:</strong><br>
🔹 הנתיב לקובץ: <code>'../../resources/sales.csv'</code><br>
🔹 הטבלה: <code>sales (date TEXT, product TEXT, category TEXT, quantity INTEGER, price INTEGER, total INTEGER)</code><br>
🔹 הקוד של מחברת 3 (<code>3_SQL_with_Python_and_CSV</code>) הוא כמעט בדיוק מה שצריך<br>
🔹 לשמירה: <code>os.makedirs('reports', exist_ok=True)</code> ו-<code>json.dump(..., ensure_ascii=False, indent=2)</code>
</p>

In [ ]:
# פתרו כאן:

# 1. יצירת הטבלה
cursor.execute('DROP TABLE IF EXISTS sales')
# cursor.execute('''CREATE TABLE sales (...)''')

# 2. קריאת הקובץ וטעינה
rows = []
with open('../../resources/sales.csv', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for record in reader:
        pass  # הוסיפו tuple לרשימה rows

# cursor.executemany('INSERT INTO sales VALUES (?, ?, ?, ?, ?, ?)', rows)
# conn.commit()

# 3+4. סיכום לפי קטגוריה ושמירה ל-JSON

## <p style="text-align: right; direction: rtl; float: right; clear: both;">🏆 אתגר: מערכת ספרייה</p>

<div class="align-center" style="display: flex; text-align: right; direction: rtl; clear: both;">
    <div style="display: flex; width: 10%; float: right; clear: both;">
        <img src="images/exercise.svg" style="height: 50px !important;" alt="תרגול">
    </div>
    <div style="width: 90%">
        <p style="text-align: right; direction: rtl; float: right; clear: both;">
        <strong>תרגול</strong>:<br>
            בנו מחלקה <code>Library</code> שמנהלת ספרייה במסד נתונים <code>library_challenge.db</code>, עם שתי טבלאות:<br>
<code>books (id, title, author, available)</code> ו-<code>loans (id, book_id, member, borrow_date, return_date)</code>.<br>
המחלקה צריכה לתמוך ב:
        </p>
    </div>
</div>

<ol style="text-align: right; direction: rtl; float: right; clear: both;">
    <li><code>add_book(title, author)</code> - הוספת ספר חדש (זמין)</li>
    <li><code>borrow(book_id, member)</code> - השאלה: אם הספר זמין, מוסיפים שורה ב-<code>loans</code> ומסמנים אותו כלא זמין. אם לא - מחזירים <code>False</code></li>
    <li><code>give_back(book_id)</code> - החזרה: מעדכנים <code>return_date</code> ומחזירים את הספר לזמין</li>
    <li><code>available_books()</code> - רשימת הספרים הזמינים</li>
    <li><code>who_has(book_id)</code> - מי מחזיק כרגע בספר (או <code>None</code>)</li>
</ol>

<p style="text-align: right; direction: rtl; float: right; clear: both;">
💡 <strong>רמזים:</strong><br>
🔹 השתמשו ב-<code>with self.conn:</code> כדי לשמור שינויים בצורה בטוחה<br>
🔹 תאריך של היום: <code>datetime.now().strftime('%Y-%m-%d')</code><br>
🔹 הוסיפו ל-<code>__init__</code> מחיקה של קובץ המסד הישן, כדי שאפשר יהיה להריץ שוב<br>
🔹 בדקו את המחלקה בעזרת הקוד שבסוף התא
</p>

In [ ]:
from datetime import datetime


class Library:
    def __init__(self, db_name='library_challenge.db'):
        if os.path.exists(db_name):
            os.remove(db_name)
        self.conn = sqlite3.connect(db_name)
        # TODO: צרו את הטבלאות books ו-loans

    def add_book(self, title, author):
        pass  # TODO

    def borrow(self, book_id, member):
        pass  # TODO: החזירו True אם ההשאלה הצליחה, ו-False אם לא

    def give_back(self, book_id):
        pass  # TODO

    def available_books(self):
        pass  # TODO: החזירו רשימה של (id, title, author)

    def who_has(self, book_id):
        pass  # TODO

    def close(self):
        self.conn.close()


# --- בדיקה ---
library = Library()
library.add_book('סיפור על אהבה וחושך', 'עמוס עוז')
library.add_book('תמול שלשום', 'ש"י עגנון')
library.add_book('עיין ערך אהבה', 'דוד גרוסמן')

print("השאלה ראשונה:", library.borrow(1, 'דנה'))     # צפוי: True
print("השאלה שנייה של אותו ספר:", library.borrow(1, 'אורי'))  # צפוי: False
print("זמינים:", library.available_books())            # צפוי: ספרים 2 ו-3
print("מי מחזיק בספר 1:", library.who_has(1))          # צפוי: דנה
library.give_back(1)
print("אחרי החזרה:", library.who_has(1))                # צפוי: None
library.close()

## <p style="text-align: right; direction: rtl; float: right; clear: both;">סגירת החיבור</p>

<p style="text-align: right; direction: rtl; float: right; clear: both;">
כשסיימתם את כל התרגילים - סוגרים את החיבור למסד הנתונים.
</p>

In [ ]:
conn.close()
print("החיבור נסגר 👋")

---

## <p style="text-align: right; direction: rtl; float: right; clear: both;">סיכום היום</p>

<ul style="text-align: right; direction: rtl; float: right; clear: both;">
    <li>ליצור טבלאות ולהכניס נתונים (<code>CREATE TABLE</code>, <code>INSERT</code>, <code>executemany</code>)</li>
    <li>לשלוף, לסנן ולמיין (<code>SELECT</code>, <code>WHERE</code>, <code>ORDER BY</code>, <code>LIMIT</code>)</li>
    <li>להעביר ערכים בבטחה עם <code>?</code></li>
    <li>לעדכן ולמחוק - תמיד עם <code>WHERE</code> (<code>UPDATE</code>, <code>DELETE</code>)</li>
    <li>לסכם קבוצות (<code>GROUP BY</code>, <code>HAVING</code>) ולחבר טבלאות (<code>INNER JOIN</code>, <code>LEFT JOIN</code>)</li>
    <li>לטעון CSV למסד נתונים ולשמור תוצאות ל-JSON / CSV</li>
</ul>

<p style="text-align: center; direction: rtl; clear: both;">
    <strong>סיימתם?</strong><br>
    מחר: Pandas - ניתוח נתונים!<br>
    <em>בהצלחה! 🐍✨</em>
</p>